## Setup

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("✓ Imports complete")

✓ Imports complete


## Configuration

In [14]:
import os
import gc
from tqdm import tqdm

START_DATE = "2025-06-01"
END_DATE = "2025-06-01"

DATA_DIR = "/home/ubuntu/data/uploads/objects/clean"
CHUNK_SIZE = 500000

def load_data(data_dir, start_date, end_date):
    """
    Load data with optimized dtypes for memory efficiency
    """
    # Define optimal dtypes upfront - saves ~50% memory
    dtypes = {
        'id': 'int32',           # Was int64 → Save 50%
        'label': 'int8',         # Values 1-8 → Save 87.5%
        'pos_x': 'float32',      # Was float64 → Save 50%
        'pos_y': 'float32',
        'pos_z': 'float32',
        'vel': 'float32',
        'vel_x': 'float32',
        'vel_y': 'float32',
        'yaw': 'float32',
        'size_x': 'float32',
        'size_y': 'float32',
        # timestamp stays int64 (needed for precision)
    }
    
    # List to collect dataframes
    dfs = []
    
    # Loop over all folders within date range
    for folder in tqdm(sorted(os.listdir(data_dir)), desc="Loading data"):
        folder_path = os.path.join(data_dir, folder)
        
        # Skip non-folders
        if not os.path.isdir(folder_path):
            continue
        
        if folder.startswith(start_date) or folder.startswith(end_date):
            # Load all parquet files inside the folder
            df_chunk = pd.read_parquet(folder_path)
            
            # Apply dtypes to matching columns
            for col, dtype in dtypes.items():
                if col in df_chunk.columns:
                    df_chunk[col] = df_chunk[col].astype(dtype)
            
            dfs.append(df_chunk)
            del df_chunk  # Clean up immediately
    
    # Combine all into single dataframe
    if dfs:
        df = pd.concat(dfs, ignore_index=True)
        # Destroy the list immediately
        del dfs
        gc.collect()
        return df
    else:
        print("No data found for given date range.")
        return pd.DataFrame()

# Usage
df = load_data(DATA_DIR, START_DATE, END_DATE)
print(f"Loaded {len(df)} records from {START_DATE} to {END_DATE}")

Loading data: 100%|██████████| 336/336 [00:01<00:00, 233.14it/s]


Loaded 39074272 records from 2025-06-01 to 2025-06-01


In [15]:
# Input paths
MDRAC_INPUT = 'results/mdrac_conflicts_new.csv'
SPF_INPUT = 'results/spf_conflicts.csv'

# Output paths
MDRAC_OUTPUT = 'results/postprocessed_conflicts/mdrac_conflicts_new.csv'
SPF_OUTPUT = 'results/postprocessed_conflicts/spf_conflicts_processed.csv'

# Filter thresholds
MAX_YAW_DIFF = 160.0  # degrees - exclude near head-on conflicts

print(f"Configuration:")
print(f"  Max yaw difference: {MAX_YAW_DIFF}°")
print(f"  M-DRAC input: {MDRAC_INPUT}")
print(f"  SPF input: {SPF_INPUT}")

Configuration:
  Max yaw difference: 160.0°
  M-DRAC input: results/mdrac_conflicts_new.csv
  SPF input: results/spf_conflicts.csv


---
## Utility Functions

In [16]:
def create_pair_id(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create normalized pair_id (always smaller ID first).
    Ensures (id1=100, id2=200) and (id1=200, id2=100) are treated as same pair.
    """
    df = df.copy()
    
    id_min = np.minimum(df['id1'].values, df['id2'].values)
    id_max = np.maximum(df['id1'].values, df['id2'].values)
    
    df['pair_id'] = id_min.astype(str) + '_' + id_max.astype(str)
    
    return df


def apply_yaw_filter(df: pd.DataFrame, max_yaw_diff: float) -> pd.DataFrame:
    """
    Filter conflicts by yaw difference threshold.
    Removes near head-on conflicts (opposite directions).
    """
    initial_count = len(df)
    filtered = df[df['yaw_diff'] < max_yaw_diff].copy()
    removed = initial_count - len(filtered)
    
    print(f"  Yaw filter (< {max_yaw_diff}°): {removed:,} removed ({removed/initial_count*100:.1f}%)")
    return filtered


def deduplicate_pairs(df: pd.DataFrame, metric_col: str) -> pd.DataFrame:
    """
    Group by pair_id and select worst case.
    
    Args:
        df: DataFrame with pair_id column
        metric_col: Column to maximize ('MDRAC' or 'composite_risk')
    
    Returns:
        Deduplicated DataFrame (one row per pair)
    """
    initial_count = len(df)
    unique_pairs = df['pair_id'].nunique()
    
    # Sort by metric (desc) and distance (asc) for tiebreaker
    df_sorted = df.sort_values([metric_col, 'dist'], ascending=[False, True])
    
    # Keep first row per pair (worst case)
    deduplicated = df_sorted.groupby('pair_id', as_index=False).first()
    
    reduction = (1 - len(deduplicated) / initial_count) * 100
    print(f"  Deduplication: {initial_count:,} → {len(deduplicated):,} ({reduction:.1f}% reduction)")
    print(f"  Avg detections per pair: {initial_count/unique_pairs:.1f}")
    
    return deduplicated


---
# M-DRAC Post-Processing

## Load M-DRAC Detections

In [17]:
print("="*70)
print("M-DRAC POST-PROCESSING")
print("="*70)

mdrac_raw = pd.read_csv(MDRAC_INPUT)
print(f"\nLoaded {len(mdrac_raw):,} raw M-DRAC detections")
print(f"Columns: {list(mdrac_raw.columns)}")
print(f"\nSample:")
mdrac_raw.head(3)

M-DRAC POST-PROCESSING

Loaded 37 raw M-DRAC detections
Columns: ['timestamp', 'id1', 'id2', 'interaction', 'leader', 'dist', 'TTC', 'MDRAC', 'closing_speed', 'speed_diff', 'yaw_diff', 'link']

Sample:


,timestamp,id1,id2,interaction,leader,dist,TTC,MDRAC,closing_speed,speed_diff,yaw_diff,link
0,2025-06-01 09:29:43.157147884,10535595,10535738,car_v_car,10535595,5.855770,1.405347,4.292573,4.166778,1.398143,157.33957,https://di-india-collab-2.flow-analytics.io/to...
1,2025-06-01 09:29:43.268170118,10535595,10535738,car_v_car,10535595,5.396849,1.299842,5.465334,4.151927,1.403953,160.69675,https://di-india-collab-2.flow-analytics.io/to...
2,2025-06-01 09:29:43.369236946,10535595,10535738,car_v_car,10535595,5.025035,1.231885,6.539496,4.079143,1.433539,163.10272,https://di-india-collab-2.flow-analytics.io/to...


In [18]:
# =============================================================================
# FILTER: Teleportation Timestamps
# =============================================================================
# Remove conflicts at timestamps where vehicles have unrealistic position jumps
# Requires original vehicle data (df) to check for teleportation

from filters.teleportation_filter import filter_conflicts_by_teleportation, MAX_JUMP_DISTANCE, SAMPLING_RATE

# Load original vehicle data if not in memory
if 'df' not in locals():
    print("Loading original vehicle data to check teleportation...")
    # TODO: Load your parquet files here
    # Example: df = pd.read_parquet('data/processed/vehicles.parquet')
    # For now, skip this filter if df not available
    print("⚠️  Skipping teleportation filter (df not loaded)")
else:
    print(f"Original vehicle data: {len(df):,} records")
    mdrac_raw = filter_conflicts_by_teleportation(
        conflicts=mdrac_raw,
        vehicle_data=df,
        max_jump=MAX_JUMP_DISTANCE,
        sampling_rate=SAMPLING_RATE,
        verbose=True
    )

Original vehicle data: 39,074,272 records

TELEPORTATION TIMESTAMP FILTER (Post-Processing)
Input conflicts: 37
Max allowed jump: 3.5 m (126.0 km/h @ 10.0Hz)


/home/ubuntu/prem/filters/teleportation_filter.py:136: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x | x.shift(-1).fillna(False) | x.shift(1).fillna(False)



Flagged timestamps in vehicle data: 19,754

Conflicts at glitchy timestamps: 0 (0.0%)
Conflicts after filtering: 37
Removed: 0 conflicts


## Apply Filters

In [28]:
print(f"\nApplying filters...")

# Step 1: Yaw filter
mdrac_filtered = apply_yaw_filter(mdrac_raw, MAX_YAW_DIFF)

# Step 2: Create pair IDs
mdrac_filtered = create_pair_id(mdrac_filtered)

# Step 3: Deduplicate by pair (keep worst MDRAC)
mdrac_processed = deduplicate_pairs(mdrac_filtered, metric_col='MDRAC')

# Remove temporary pair_id column
mdrac_output = mdrac_processed.drop(columns=['pair_id'])

print(f"\n✓ M-DRAC processing complete")
print(f"  Final conflicts: {len(mdrac_output):,}")
print(f"  Total reduction: {(1 - len(mdrac_output)/len(mdrac_raw))*100:.1f}%")


Applying filters...
  Yaw filter (< 160.0°): 12 removed (32.4%)
  Deduplication: 25 → 12 (52.0% reduction)
  Avg detections per pair: 2.1

✓ M-DRAC processing complete
  Final conflicts: 12
  Total reduction: 67.6%


## M-DRAC Analysis

In [29]:
print("\n" + "="*70)
print("M-DRAC ANALYSIS")
print("="*70)

print("\nMDRAC Statistics:")
print(mdrac_output['MDRAC'].describe())

print("\nInteraction Type Distribution:")
print(mdrac_output['interaction'].value_counts())

print("\nTop 10 Highest MDRAC Conflicts:")
top10_mdrac = mdrac_output.nlargest(10, 'MDRAC')[[
    'timestamp', 'id1', 'id2', 'interaction', 'leader', 
    'dist', 'TTC', 'MDRAC', 'yaw_diff'
]]
print(top10_mdrac.to_string(index=False))


M-DRAC ANALYSIS

MDRAC Statistics:
count    12.000000
mean           inf
std            NaN
min       3.632487
25%      10.796930
50%      86.238603
75%            inf
max            inf
Name: MDRAC, dtype: float64

Interaction Type Distribution:
interaction
car_v_car      11
car_v_truck     1
Name: count, dtype: int64

Top 10 Highest MDRAC Conflicts:
                    timestamp      id1      id2 interaction   leader     dist      TTC      MDRAC   yaw_diff
2025-06-01 09:35:30.561595917 10540034 10540130   car_v_car 10540130 4.248736 0.341212        inf 159.046140
2025-06-01 14:57:40.383486986 10788941 10788955   car_v_car 10788941 3.408543 0.654531        inf   5.595291
2025-06-01 18:20:33.385298967 10936238 10936713 car_v_truck 10936238 4.575431 0.589487        inf  85.998604
2025-06-01 14:59:43.533787966 10789621 10790156   car_v_car 10789621 6.888694 0.930858 340.777220 159.130070
2025-06-01 19:29:42.062628984 10981489 10981635   car_v_car 10981489 6.152759 0.948025 115.788944 15

/home/ubuntu/prem/.venv/lib/python3.13/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


## Save M-DRAC Results

In [30]:
mdrac_output.to_csv(MDRAC_OUTPUT, index=False)
print(f"\n✓ Saved M-DRAC processed conflicts to: {MDRAC_OUTPUT}")
print(f"  Rows: {len(mdrac_output):,}")


✓ Saved M-DRAC processed conflicts to: results/postprocessed_conflicts/mdrac_conflicts_new.csv
  Rows: 12


---
# SPF Post-Processing

## Load SPF Detections

In [22]:
print("\n" + "="*70)
print("SPF POST-PROCESSING")
print("="*70)

spf_raw = pd.read_csv(SPF_INPUT)
print(f"\nLoaded {len(spf_raw):,} raw SPF detections")
print(f"Columns: {list(spf_raw.columns)}")
print(f"\nSample:")
spf_raw.head(3)


SPF POST-PROCESSING

Loaded 982 raw SPF detections
Columns: ['timestamp', 'id1', 'id2', 'interaction', 'dist', 'TTC', 'composite_risk', 'closing_speed', 'speed_diff', 'yaw_diff', 'link']

Sample:


,timestamp,id1,id2,interaction,dist,TTC,composite_risk,closing_speed,speed_diff,yaw_diff,link
0,2025-06-01 00:07:50.832639933,10287921,10288098,car_v_car,3.781766,0.914998,0.969074,4.133087,3.848074,2.461928,https://di-india-collab.flow-analytics.io/tool...
1,2025-06-01 00:07:50.941036940,10287921,10288098,car_v_car,3.313090,0.762758,0.989730,4.343566,3.944590,5.371479,https://di-india-collab.flow-analytics.io/tool...
2,2025-06-01 00:07:51.038043976,10287921,10288098,car_v_car,2.912292,0.642655,0.993468,4.531657,4.047575,6.266725,https://di-india-collab.flow-analytics.io/tool...


In [23]:
# =============================================================================
# FILTER: Teleportation Timestamps
# =============================================================================
# Remove conflicts at timestamps where vehicles have unrealistic position jumps

from filters.teleportation_filter import filter_conflicts_by_teleportation, MAX_JUMP_DISTANCE, SAMPLING_RATE

# Use same vehicle data from M-DRAC section
if 'df' not in locals():
    print("⚠️  Skipping teleportation filter (df not loaded)")
else:
    print(f"Original vehicle data: {len(df):,} records")
    spf_raw = filter_conflicts_by_teleportation(
        conflicts=spf_raw,
        vehicle_data=df,
        max_jump=MAX_JUMP_DISTANCE,
        sampling_rate=SAMPLING_RATE,
        verbose=True
    )

Original vehicle data: 39,074,272 records

TELEPORTATION TIMESTAMP FILTER (Post-Processing)
Input conflicts: 982
Max allowed jump: 3.5 m (126.0 km/h @ 10.0Hz)


/home/ubuntu/prem/filters/teleportation_filter.py:136: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x | x.shift(-1).fillna(False) | x.shift(1).fillna(False)



Flagged timestamps in vehicle data: 19,754

Conflicts at glitchy timestamps: 0 (0.0%)
Conflicts after filtering: 982
Removed: 0 conflicts


## Apply Filters

In [24]:
print(f"\nApplying filters...")

# Step 1: Yaw filter
spf_filtered = apply_yaw_filter(spf_raw, MAX_YAW_DIFF)

# Step 2: Create pair IDs
spf_filtered = create_pair_id(spf_filtered)

# Step 3: Deduplicate by pair (keep highest composite_risk)
spf_processed = deduplicate_pairs(spf_filtered, metric_col='composite_risk')

# Remove temporary pair_id column
spf_output = spf_processed.drop(columns=['pair_id'])

print(f"\n✓ SPF processing complete")
print(f"  Final conflicts: {len(spf_output):,}")
print(f"  Total reduction: {(1 - len(spf_output)/len(spf_raw))*100:.1f}%")


Applying filters...
  Yaw filter (< 160.0°): 4 removed (0.4%)
  Deduplication: 978 → 146 (85.1% reduction)
  Avg detections per pair: 6.7

✓ SPF processing complete
  Final conflicts: 146
  Total reduction: 85.1%


## SPF Analysis

In [25]:
print("\n" + "="*70)
print("SPF ANALYSIS")
print("="*70)

print("\nComposite Risk Statistics:")
print(spf_output['composite_risk'].describe())

print("\nInteraction Type Distribution:")
print(spf_output['interaction'].value_counts())

# print("\nTop 10 Highest Risk Conflicts:")
# top10_spf = spf_output.nlargest(10, 'composite_risk')[[
#     'timestamp', 'id1', 'id2', 'interaction', 
#     'dist', 'TTC', 'composite_risk', 'yaw_diff'
# ]]
# print(top10_spf.to_string(index=False))


SPF ANALYSIS

Composite Risk Statistics:
count    146.000000
mean       0.703251
std        0.142525
min        0.500032
25%        0.589718
50%        0.672457
75%        0.806464
max        0.999049
Name: composite_risk, dtype: float64

Interaction Type Distribution:
interaction
car_v_car      129
van_v_car        9
car_v_van        5
van_v_van        1
van_v_truck      1
car_v_truck      1
Name: count, dtype: int64


## Save SPF Results

In [26]:
spf_output.to_csv(SPF_OUTPUT, index=False)
print(f"\n✓ Saved SPF processed conflicts to: {SPF_OUTPUT}")
print(f"  Rows: {len(spf_output):,}")


✓ Saved SPF processed conflicts to: results/postprocessed_conflicts/spf_conflicts_processed.csv
  Rows: 146


---
# Summary Statistics

In [27]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

summary = pd.DataFrame({
    'Method': ['M-DRAC', 'SPF'],
    'Raw Detections': [len(mdrac_raw), len(spf_raw)],
    'After Yaw Filter': [len(mdrac_filtered), len(spf_filtered)],
    'After Deduplication': [len(mdrac_output), len(spf_output)],
    'Total Reduction %': [
        f"{(1 - len(mdrac_output)/len(mdrac_raw))*100:.1f}%",
        f"{(1 - len(spf_output)/len(spf_raw))*100:.1f}%"
    ]
})

print("\n", summary.to_string(index=False))


FINAL SUMMARY

 Method  Raw Detections  After Yaw Filter  After Deduplication Total Reduction %
M-DRAC              37                25                   12             67.6%
   SPF             982               978                  146             85.1%
